<a href="https://colab.research.google.com/github/adithya95978/medicore/blob/main/medical_chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Run this command in a Google Colab cell
!pip list


Package                                  Version
---------------------------------------- -------------------
absl-py                                  1.4.0
absolufy-imports                         0.3.1
accelerate                               1.10.1
aiofiles                                 24.1.0
aiohappyeyeballs                         2.6.1
aiohttp                                  3.12.15
aiohttp-retry                            2.9.1
aiolimiter                               1.2.1
aiosignal                                1.4.0
alabaster                                1.0.0
albucore                                 0.0.24
albumentations                           2.0.8
ale-py                                   0.11.2
alembic                                  1.16.5
altair                                   5.5.0
annotated-types                          0.7.0
antlr4-python3-runtime                   4.9.3
anyio                                    4.11.0
anywidget                           

In [ ]:
!pip install langchain
!pip install langchain-core
!pip install  langchain-community
!pip install  langchain-groq
!pip install  pypdf
!pip install langchain-pinecone
!pip install  voyageai
!pip install python-dotenv
!pip install tqdm


In [ ]:
import os
from google.colab import userdata

# Load API keys from Colab secrets
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
os.environ["PINECONE_API_KEY"] = userdata.get('PINECONE_API_KEY')
os.environ["VOYAGE_API_KEY"] = userdata.get('VOYAGE_API_KEY')

print("API Keys loaded successfully!")

API Keys loaded successfully!


In [ ]:
from google.colab import files
import os

print("Please upload your PDF document:")
uploaded = files.upload()

if not uploaded:
    print("No file was uploaded. Please run the cell again.")
else:
    # Get the name of the uploaded file
    file_path = list(uploaded.keys())[0]
    print(f"\nSuccessfully uploaded '{file_path}'")

Please upload your PDF document:


Saving Cardiovascular-Pathophysiology-for-Pre-Clinical-Students-Binks-Andrew-P..pdf to Cardiovascular-Pathophysiology-for-Pre-Clinical-Students-Binks-Andrew-P. (1).pdf

Successfully uploaded 'Cardiovascular-Pathophysiology-for-Pre-Clinical-Students-Binks-Andrew-P. (1).pdf'


In [ ]:
import time
from pathlib import Path
from tqdm.auto import tqdm
from pinecone import Pinecone, ServerlessSpec
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings.base import Embeddings
from langchain_pinecone import PineconeVectorStore
import voyageai

# --- Configuration from your file ---
PINECONE_INDEX_NAME = "medicalindex"
PINECONE_ENV = "us-east-1"

# --- Custom VoyageAI Embeddings Class from your file ---
class VoyageAIEmbeddings(Embeddings):
    def __init__(self, model_name="voyage-large-2-instruct"):
        self.client = voyageai.Client(api_key=os.getenv("VOYAGE_API_KEY"))
        self.model_name = model_name

    def embed_documents(self, texts):
        response = self.client.embed(texts, model=self.model_name, input_type="document")
        return response.embeddings

    def embed_query(self, query):
        response = self.client.embed([query], model=self.model_name, input_type="query")
        return response.embeddings[0]

# --- Pinecone Setup from your file ---
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
spec = ServerlessSpec(cloud="aws", region=PINECONE_ENV)

if PINECONE_INDEX_NAME not in pc.list_indexes().names():
    print(f"Creating new Pinecone index: {PINECONE_INDEX_NAME}")
    pc.create_index(name=PINECONE_INDEX_NAME, dimension=1024, metric="dotproduct", spec=spec)
    while not pc.describe_index(PINECONE_INDEX_NAME).status["ready"]:
        time.sleep(1)
else:
    print(f"Using existing Pinecone index: {PINECONE_INDEX_NAME}")

index = pc.Index(PINECONE_INDEX_NAME)

# --- Document Processing Logic from your file ---
print("\nProcessing the document...")
embed_model = VoyageAIEmbeddings()
loader = PyPDFLoader(file_path)
documents = loader.load()
print(f"Loaded {len(documents)} pages.")

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(documents)
print(f"Split document into {len(chunks)} chunks.")

# Batch processing
batch_size = 8
for i in tqdm(range(0, len(chunks), batch_size), desc=f"Processing {Path(file_path).name}"):
    batch_chunks = chunks[i:i + batch_size]
    texts = [chunk.page_content for chunk in batch_chunks]
    metadatas = [chunk.metadata for chunk in batch_chunks]
    ids = [f"{Path(file_path).stem}-{i+j}" for j in range(len(batch_chunks))]

    embeddings = embed_model.embed_documents(texts)
    index.upsert(vectors=zip(ids, embeddings, metadatas))

    # --- ADD THIS DELAY ---
    # To respect the 3 RPM limit, wait 20 seconds between each request.
    print("Waiting 20 seconds to respect rate limit...")
    time.sleep(20)

print(f"\n✅ Upload complete for {file_path}. Knowledge base is ready.")

Using existing Pinecone index: medicalindex

Processing the document...
Loaded 79 pages.
Split document into 346 chunks.


Processing Cardiovascular-Pathophysiology-for-Pre-Clinical-Students-Binks-Andrew-P. (1).pdf:   0%|          | …

Waiting 20 seconds to respect rate limit...
Waiting 20 seconds to respect rate limit...
Waiting 20 seconds to respect rate limit...
Waiting 20 seconds to respect rate limit...
Waiting 20 seconds to respect rate limit...
Waiting 20 seconds to respect rate limit...
Waiting 20 seconds to respect rate limit...
Waiting 20 seconds to respect rate limit...
Waiting 20 seconds to respect rate limit...
Waiting 20 seconds to respect rate limit...
Waiting 20 seconds to respect rate limit...
Waiting 20 seconds to respect rate limit...
Waiting 20 seconds to respect rate limit...
Waiting 20 seconds to respect rate limit...
Waiting 20 seconds to respect rate limit...
Waiting 20 seconds to respect rate limit...
Waiting 20 seconds to respect rate limit...
Waiting 20 seconds to respect rate limit...
Waiting 20 seconds to respect rate limit...
Waiting 20 seconds to respect rate limit...
Waiting 20 seconds to respect rate limit...
Waiting 20 seconds to respect rate limit...
Waiting 20 seconds to respect ra

In [ ]:
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

In [ ]:
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA
from langchain_groq import ChatGroq

# --- Logic from your llm.py file ---
def get_llm_chain(retriever):
    # All lines inside this function should have the same starting indentation.
# This is the corrected line
    # This is the corrected line

    llm=ChatGroq(groq_api_key=os.environ["GROQ_API_KEY"],model_name="llama-3.3-70b-versatile")

    prompt = PromptTemplate(
        input_variables=["context", "question"],
        template="""
You are **MediBot**, an AI-powered assistant trained to help users understand medical documents and health-related questions.
Your job is to provide clear, accurate, and helpful responses based **only on the provided context**.
---
🔍 **Context**:
{context}

🙋‍♂️ **User Question**:
{question}
---
💬 **Answer**:
- Respond in a calm, factual, and respectful tone.
- Use simple explanations when needed.
- If the context does not contain the answer, say: "I'm sorry, but I couldn't find relevant information in the provided documents."
- Do NOT make up facts.
- Do NOT give medical advice or diagnoses.
"""
    )

    return RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=retriever,
        chain_type_kwargs={"prompt": prompt},
        return_source_documents=True
    )

# --- Create the retriever and the QA chain ---
print("Initializing the QA chain...")
# We need to create a LangChain VectorStore object to build a retriever
vectorstore = PineconeVectorStore(index_name=PINECONE_INDEX_NAME, embedding=embed_model)
retriever = vectorstore.as_retriever()
qa_chain = get_llm_chain(retriever)
print("Chain is ready.")

# --- Ask your question ---
my_question = "Premature Ventricular Contractions"
print(f"\n❓ Your Question: {my_question}")
print("\n⏳ Getting your answer...")

# Invoke the chain using the format it expects
result = qa_chain.invoke({"query": my_question})

# --- Display the results ---
print("\n✅ Final Answer:")
print(result["result"])

print("\n\n📄 Sources Used:")
for doc in result["source_documents"]:
    print(f"- Source: {doc.metadata.get('source', 'N/A')}, Page: {doc.metadata.get('page', 'N/A')}")

Initializing the QA chain...
Chain is ready.

❓ Your Question: Premature Ventricular Contractions

⏳ Getting your answer...



✅ Final Answer:
Premature Ventricular Contractions (PVCs) are a type of irregular heartbeat. They occur when the ventricles, which are the lower chambers of the heart, contract too soon, before the heart has fully filled with blood. This can cause the heart to feel like it's skipping a beat or beating irregularly. PVCs are relatively common and can be felt as a skipped beat or a flutter in the chest. If you're concerned about PVCs or any other heart-related issue, it's best to consult a healthcare professional for proper evaluation and guidance.


📄 Sources Used:
